In [2]:
import sys
sys.path.insert(0, 'AstroM3')

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import random
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np
import joblib
from scipy.interpolate import interp1d
from datetime import datetime
from scipy import stats
import torch
import torch.nn as nn
import torch.nn.functional as F

from AstroM3.core.dataset import DataGenerator
from AstroM3.core.model import GalSpecNet

In [26]:
class BTSModel(nn.Module):
    def __init__(self, config):
        super(BTSModel, self).__init__()

        self.classification = True if config['mode'] == 'image' else False
        
        self.block1 = nn.Sequential(
            nn.Conv2d(
                in_channels=config['input_channels'], 
                out_channels=config['conv1_channels'], 
                kernel_size=config['conv_kernel'],
                padding='same'
            ),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=config['conv1_channels'], 
                out_channels=config['conv1_channels'], 
                kernel_size=config['conv_kernel'], 
                padding='same'
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(config['conv_dropout1'])
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
                in_channels=config['conv1_channels'], 
                out_channels=config['conv2_channels'], 
                kernel_size=config['conv_kernel'],
                padding='same'
            ),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=config['conv2_channels'], 
                out_channels=config['conv2_channels'], 
                kernel_size=config['conv_kernel'],
                padding='same'
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=4),
            nn.Dropout(config['conv_dropout2'])
        )

        if self.classification:
            self.fc = nn.Linear(1024, config['num_classes'])
        
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.shape[0], -1)
        
        if self.classification:
            x = self.fc(x)
        
        return x

In [34]:
config = {
    'mode': 'image',
    'num_classes': 10,
    'input_channels': 3,
    "conv1_channels": 64,
    "conv2_channels": 16,
    "conv_kernel": 3,
    "conv_dropout1": 0.45,
    "conv_dropout2": 0.65,
}

In [35]:
model = BTSModel(config)

In [36]:
image = torch.rand((1, 3, 64, 64))

In [37]:
with torch.no_grad():
    out = model(image)

In [38]:
out.shape

torch.Size([1, 10])